## XBRL US API - Python example  
This sample Python code queries assertion data for SEC filings.

### Authenticate for access token 
Click the run button and enter your XBRL US Web account email, account password, Client ID, and secret (get these from https://xbrl.us/access-token), pressing the Enter key on the keyboard after each entry.

XBRL US limits records returned for a query to improve efficiency; this script loops to collect all data from the Public Filings Database for a query. **Non-members might not be able to return all data for a query** - join XBRL US for comprehensive access - https://xbrl.us/join.

In [2]:
# @title
import os, re, sys, json
import requests
import pandas as pd
from IPython.display import display, HTML
import numpy as np
import getpass
from datetime import datetime
import urllib
from urllib.parse import urlencode

api = input('Enter subdomain ("api" or left blank to query public results, otherwise enter a value) ') or 'api'
baseurl = 'https://' + api + '.xbrl.us/'

class tokenInfoClass:
    access_token = None
    refresh_token = None
    email = None
    username = None
    client_id = None
    client_secret = None
    authurl = baseurl + 'oauth2/token'
    headers = {"Content-Type": "application/x-www-form-urlencoded"}

def refresh(info):
    refresh_auth = {
                'client_id': info.client_id,
                'client_secret' : info.client_secret,
                'grant_type' : 'refresh_token',
                'platform' : 'ipynb',
                'refresh_token' : info.refresh_token
                }
    refreshres = requests.post(info.authurl, data=refresh_auth, headers=info.headers)
    refresh_json = refreshres.json()
    info.access_token = refresh_json.get('access_token')
    info.refresh_token = refresh_json.get('refresh_token')
    print('Your access token(%s) is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.' % (info.access_token))
    return info

tokenInfo = tokenInfoClass()

# Helper to prompt only if value is missing
def prompt_if_missing(value, prompt_text, secret=False):
    if value:
        return value
    if secret:
        return getpass.getpass(prompt=prompt_text)
    return input(prompt_text)

# Load creds.json (if exists) and ask for user confirmation
creds = {}
use_creds_file = False
if os.path.exists('creds.json'):
    try:
        with open('creds.json', 'r') as f:
            creds = json.load(f)
        confirm = input("Would you like to use credentials on file for this? (yes/no): ").strip().lower()
        use_creds_file = confirm in ('yes', 'y')
        if use_creds_file:
            print("Using credentials on file")
        else:
            print("Enter credentials manually")
            creds = {}
    except Exception as e:
        print("Warning: failed to read creds.json:", e)
        creds = {}

use_test = 'test' in api.lower()

if creds and use_creds_file:
    # Determine credential source: prod > test > top-level > prompts
    selected = None
    
    # First, try nested objects if they exist (e.g., { "prod": {...}} or { "test": {...}})
    if use_test and isinstance(creds.get('test'), dict):
        selected = creds['test']
    elif not use_test and isinstance(creds.get('prod'), dict):
        selected = creds['prod']
    
    # Next, try prefixed keys (prod*/test* if requested)
    if not selected:
        selected = {}
        keys = ['email', 'password', 'client_id', 'client_secret']
        prefix = 'test' if use_test else 'prod'
        for k in keys:
            prefixed_key = prefix + k
            if creds.get(prefixed_key):
                selected[k] = creds.get(prefixed_key)
            # fall back to top-level key if prefix variant not found
            elif creds.get(k):
                selected[k] = creds.get(k)
    
    # Verify we have all required keys
    if not all(selected.get(k) for k in ('email', 'password', 'client_id', 'client_secret')):
        # Fill in missing values from prompts
        selected = {
            'email': selected.get('email'),
            'password': selected.get('password'),
            'client_id': selected.get('client_id'),
            'client_secret': selected.get('client_secret')
        }
    
    # Assign values, prompting for any missing ones
    tokenInfo.email = prompt_if_missing(selected.get('email'), 'Enter your XBRL US Web account email: ')
    tokenInfo.password = prompt_if_missing(selected.get('password'), 'Password: ', secret=True)
    tokenInfo.client_id = prompt_if_missing(selected.get('client_id'), 'Client ID: ', secret=True)
    tokenInfo.client_secret = prompt_if_missing(selected.get('client_secret'), 'Secret: ', secret=True)

    cred_source = 'test credentials' if use_test else 'prod credentials'
    print(f'Using {cred_source}')
else:
    # No creds.json or user declined — prompt the user
    tokenInfo.email = input('Enter your XBRL US Web account email: ')
    tokenInfo.password = getpass.getpass(prompt='Password: ')
    tokenInfo.client_id = getpass.getpass(prompt='Client ID: ')
    tokenInfo.client_secret = getpass.getpass(prompt='Secret: ')

body_auth = {'username' : tokenInfo.email,
            'client_id': tokenInfo.client_id,
            'client_secret' : tokenInfo.client_secret,
            'password' : tokenInfo.password,
            'grant_type' : 'password',
            'platform' : 'ipynb' }

# Make auth request
payload = urlencode(body_auth)
res = requests.request("POST", tokenInfo.authurl, data=payload, headers=tokenInfo.headers)
auth_json = res.json()

if 'error' in auth_json:
    print("\n\nThere was a problem generating the access token: %s  Run the first cell again and enter the credentials." % (auth_json.get('error_description', auth_json)))
else:
    tokenInfo.access_token = auth_json.get('access_token')
    tokenInfo.refresh_token = auth_json.get('refresh_token')
    if tokenInfo.access_token and tokenInfo.refresh_token:
        print ("\n\nYour access token expires in 60 minutes. After it expires, it should be regenerated automatically.  If not, run the cell rerun the first query cell. \n\nFor now, skip ahead to the section 'Make a Query'.")
    else:
        print("\n\nAuthentication completed but tokens were not returned. Response: {}".format(auth_json))

#print(vars(tokenInfo))
if tokenInfo.access_token and tokenInfo.refresh_token:
    print('\n\naccess token: ' + tokenInfo.access_token + ' refresh token: ' + tokenInfo.refresh_token)
else:
    print('\n\nNo access token was generated. Check the messages above for errors.')

Using credentials on file
Using prod credentials


Your access token expires in 60 minutes. After it expires, it should be regenerated automatically.  If not, run the cell rerun the first query cell. 

For now, skip ahead to the section 'Make a Query'.


access token: 313f521b-e3a6-4a63-a5b8-6c471a9d45b5 refresh token: b36ba6ce-2d19-41ea-b0c5-f966948ee99a


### Make a query 
After the access token confirmation appears above, you can modify the code below to update the query, then run the cell to save it. In the next cell click run to query for results.
  
Refer to XBRL API documentation at https://xbrlus.github.io/xbrl-api/#/assertion/getAssertionDetails for other endpoints and parameters to filter and return. 

In [3]:
# Define the parameters of the query - this query returns DQC assertions for specified years

endpoint = 'assertion'
XBRL_Elements = [
'0176'
    ]
report_year = [
    'ifrs 2026',
    'ifrs 2025',
    'ifrs 2024',
    'ifrs 2023',
    'us gaap 2026',
    'us gaap 2025',
    'us gaap 2024',
    'us gaap 2023'
    ]
fields = [ 
     # this is the list of the characteristics of the data being returned by the query
    'assertion.code.sort(ASC)',
    'report.base-taxonomy.sort(DESC)',
    'report.document-type',
    'entity.name',
    'entity.code',
    'report.accession',
    'assertion.run-date',
    'assertion.type',
    'report.accepted-timestamp.sort(DESC)',
    'assertion.detail',
    #'report.entry-url',
    #'assertion.limit()'
    ]

# Set unique rows as True of False (True drops any duplicate rows)
unique = True

# Limit the number of rows displayed by the notebook (does not impact the data frame)
rows_to_display = 2 # Set as '' to display all rows in the notebook

# Below is the list of what's being queried using the search endpoint.
 
params = { 
    'assertion.type': ','.join(XBRL_Elements), 
    #'report.base-taxonomy': ','.join(report_year),
    'fields': ','.join(fields)
    }

print('\n\nclick the run button below to execute this query')



click the run button below to execute this query


In [4]:
# @title
# ### Execute the query with loop for all results 
### THIS SECTION DOES NOT NEED TO BE EDITED

search_endpoint = baseurl + 'api/v1/' + endpoint + '/search'
if unique:
    search_endpoint += "?unique"
orig_fields = params['fields']
offset_value = 0
res_df = []
count = 0
query_start = datetime.now()
printed = False
run_query = True

while True:
    if not printed:
        print("On", query_start.strftime("%c"), tokenInfo.email, "(client ID:", str(tokenInfo.client_id.split('-')[0]), "...) started the query and")
        printed = True
    retry = 0
    while retry < 3:
        res = requests.get(search_endpoint, params=params, headers={'Authorization' : 'Bearer {}'.format(tokenInfo.access_token)})
        res_json = res.json()
        if 'error' in res_json:
            if res_json['error_description'] == 'Bad or expired token':
                tokenInfo = refresh(tokenInfo)
            else: 
                print('There was an error: {}'.format(res_json['error_description']))
                run_query = False
                break
        else: 
		        break
        retry +=1
        if retry >= 3:
            print("Can't refresh the access token.  Run the first query block, then rerun the query.")
            run_query = False

    if not run_query:
       break

    print("up to", str(offset_value + res_json['paging']['limit']), "records are found so far ...")

    res_df += res_json['data']

    if res_json['paging']['count'] < res_json['paging']['limit']:
        print(" - this set contained fewer than the", res_json['paging']['limit'], "possible, only", str(res_json['paging']['count']), "records.")
        break
    else: 
        offset_value += res_json['paging']['limit'] 
        if 100 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 10 * res_json['paging']['limit']:
                        break 
        elif 500 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 4 * res_json['paging']['limit']:
                        break 
        params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)

if not 'error' in res_json:
    current_datetime = datetime.now().replace(microsecond=0)
    time_taken = current_datetime - query_start
    index = pd.DataFrame(res_df).index
    total_rows = len(index)
    your_limit = res_json['paging']['limit']
    limit_message = "If the results below match the limit noted above, you might not be seeing all rows, and should consider upgrading (https://xbrl.us/access-token).\n"
    
    if your_limit == 100:
        print("\nThis non-Member account has a limit of " , 10 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    elif your_limit == 500:
        print("\nThis Basic Individual Member account has a limit of ", 4 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    
    print("\nAt " + current_datetime.strftime("%c") +  ", the query finished with  ", str(total_rows), "  rows returned in " + str(time_taken) + " for \n" +  urllib.parse.unquote(res.url))
    
    df = pd.DataFrame(res_df)
    # the format truncates the HTML display of numerical values to two decimals; .csv data is unaffected
    pd.options.display.float_format = '{:,.2f}'.format
    display(HTML(df.to_html(max_rows=rows_to_display)))

On Tue Oct 20 13:05:02 2026 david.tauriello@xbrl.us (client ID: 33cccd22 ...) started the query and
up to 5000 records are found so far ...
up to 10000 records are found so far ...
up to 15000 records are found so far ...
 - this set contained fewer than the 5000 possible, only 4208 records.

At Tue Oct 20 13:05:07 2026, the query finished with   14208   rows returned in 0:00:04.436864 for 
https://api.xbrl.us/api/v1/assertion/search?unique&assertion.type=0176&fields=assertion.code.sort(ASC),report.base-taxonomy.sort(DESC),report.document-type,entity.name,entity.code,report.accession,assertion.run-date,assertion.type,report.accepted-timestamp.sort(DESC),assertion.detail,assertion.offset(10000)


,assertion.code,report.base-taxonomy,report.document-type,entity.name,entity.code,report.accession,assertion.run-date,assertion.type,report.accepted-timestamp,assertion.detail
0,DQC.IFRS.0176.10118,US GAAP 2024,424B2,Main Street Capital Corporation,0001396440,0001396440-24-000112,2024-05-29,0176,2024-05-29 16:11:00,"2024-05-29 17:04:32,861 [DQC.IFRS.0176.10118] The filer has used the concept us-gaap:NetAssetValuePerShare with a value of 21.52 in the filing. This is a us-gaap element and should be replaced with an IFRS element.\n\nThe properties of this us-gaap:NetAssetValuePerShare fact are:\nPeriod :2020-09-30\nDimensions : \n\nRule Element Id:10118\nRule version: 23.0.12 - ck0001396440-20240529.htm 8"
...,...,...,...,...,...,...,...,...,...,...
14207,DQC.IFRS.0176.10119,IFRS 2022,20-F/A,SNDL Inc.,0001766600,0000950170-24-034351,2024-03-25,0176,2024-03-20 19:25:00,"2024-03-25 16:38:43,800 [DQC.IFRS.0176.10119] The filer has used the US-GAAP concept us-gaap:RestrictedCashAbstract in the presentation tree of the filing please choose an IFRS concept as a replacement.\n\nRule Element Id:10119\nRule version: 23.0.7 - sndl-20221231.htm 19"


The cell below will save the initial dataframe to a local file or Google Drive as a .csv

In [11]:
# If you run this program locally, you can save the output to a file 
# on your computer (modify D:\results.csv to your system)

df.to_csv(r"D:\dqc_0176.csv",sep=",")

# Google Colab users - comment out the line above and uncomment the code below to save the data frame as a .csv in your Google Drive

#from google.colab import drive
#drive.mount('drive')
#df.to_csv('dqc_0176.csv')
#!cp data.csv "drive/My Drive/"

The code below filters the dataframe by beginning and ending times for SEC filings, then summarizes the dataframe by consolidating assertion code by major rule and adding a column summarizing the number of unique filings for each assertion code. **Click the run button to execute the filtering and grouping.** 

In [ ]:
# @title
# Convert 'report.accepted-timestamp' to datetime if it's not already
df['report.accepted-timestamp'] = pd.to_datetime(df['report.accepted-timestamp'])

# Define the calendar year and quarters to filter on for 2026 (e.g. [1], [1, 2, 3], or [1, 2, 3, 4])
year = 2026
quarters = [1, 2, 3]

# Build a mask that is True for any row falling in one of the selected quarters
in_selected_quarters = pd.Series(False, index=df.index)
for q in quarters:
    period = pd.Period(f'{year}Q{q}', freq='Q')
    in_selected_quarters |= (df['report.accepted-timestamp'] >= period.start_time) & (df['report.accepted-timestamp'] <= period.end_time)

# Filter the dataframe to the selected calendar quarters
filtered_df = df[in_selected_quarters].copy()

# Label each row with its calendar quarter so results can be broken out per quarter
filtered_df.insert(0, 'quarter', filtered_df['report.accepted-timestamp'].dt.quarter.apply(lambda q: f'{year}Q{q}'))

# Count the occurrences of each assertion.code, broken out by quarter
assertion_code_counts = filtered_df.groupby(['quarter', 'assertion.code']).agg(
    count=('assertion.code', 'size'),
    filings=('report.accession', 'nunique')
).reset_index()

# Sort the assertion_code_counts table by quarter, then 'count' in descending order
assertion_code_counts = assertion_code_counts.sort_values(by=['quarter', 'count'], ascending=[True, False])

# Display the table
display(HTML(assertion_code_counts.to_html(index=False)))

# Sum how many times each assertion.code occurs within the same accession
filtered_df.insert(0, 'error_count_in_accession', filtered_df.groupby(['report.accession', 'assertion.code'])['assertion.code'].transform('size'))

# Collapse to at most one row per accession per assertion.code (DQC.IFRS.0176.10118 / DQC.IFRS.0176.10119),
# so each accession shows at most two rows, with error_count_in_accession as the left-most column
accession_detail_df = filtered_df.drop_duplicates(subset=['report.accession', 'assertion.code']).sort_values(
    by=['quarter', 'error_count_in_accession'], ascending=[True, False]
)

# Display the collapsed dataframe, dropping assertion.detail from the displayed table only
display(HTML(accession_detail_df.drop(columns=['assertion.detail'], errors='ignore').to_html(max_rows=rows_to_display)))

quarter,assertion.code,count,filings
2026Q1,DQC.IFRS.0176.10118,2847,5
2026Q1,DQC.IFRS.0176.10119,798,34
2026Q2,DQC.IFRS.0176.10118,1400,8
2026Q2,DQC.IFRS.0176.10119,654,66
2026Q3,DQC.IFRS.0176.10119,35,22
2026Q3,DQC.IFRS.0176.10118,1,1


,error_count_in_accession,quarter,assertion.code,report.base-taxonomy,report.document-type,entity.name,entity.code,report.accession,assertion.run-date,assertion.type,report.accepted-timestamp
2582,839,2026Q1,DQC.IFRS.0176.10118,IFRS 2024,20-F/A,"ZenaTech, Inc.",0001997403,0001376474-26-000218,2026-03-07,0176,2026-02-27 17:12:00
...,...,...,...,...,...,...,...,...,...,...,...
10337,1,2026Q3,DQC.IFRS.0176.10119,IFRS 2025,20-F,VCI Global Limited,0001930510,0001213900-26-078044,2026-07-15,0176,2026-07-14 18:17:00


In [13]:
# If you run this program locally, you can save the output to a file 
# on your computer (modify D:\results.csv to your system)

accession_detail_df.to_csv(r"D:\accession-filing-detail.csv", sep=",", index=False)

# Google Colab users - comment out the line above and uncomment the code below to save the data frame as a .csv in your Google Drive

#from google.colab import drive
#drive.mount('drive')
#accession_detail_df.to_csv('accession-filing-detail.csv')
#!cp data.csv "drive/My Drive/"